# Real-Time Sign Language Recognition (ASL alphabet)

Elevate Labs AI & ML Internship - Project Phase, Project 7.

**Idea:** MediaPipe finds 21 hand landmarks in an image; a small neural network (Keras) learns which letter those landmarks represent. The trained model is then used on a live webcam (see `realtime_sign_recognition.py`).

Run this notebook in Google Colab (Runtime -> Run all, after adding your Kaggle token).

## 1. Install and import MediaPipe

In [ ]:
!pip install -q mediapipe
import mediapipe as mp
print(mp.__version__)

## 2. Download the MediaPipe hand model and create the detector
Recent MediaPipe versions no longer include `mp.solutions`, so the Tasks API (`HandLandmarker`) is used.

In [ ]:
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task
!ls -lh hand_landmarker.task

In [ ]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)
detector = HandLandmarker.create_from_options(options)

## 3. Feature function: hand image -> 63 numbers
21 landmarks x (x, y, z). Points are made relative to the wrist and divided by the largest distance, so hand position and size in the picture do not matter.

In [ ]:
import numpy as np

def get_landmarks(image_path):
    image = mp.Image.create_from_file(image_path)
    result = detector.detect(image)
    if len(result.hand_landmarks) == 0:
        return None
    hand = result.hand_landmarks[0]
    points = np.array([[p.x, p.y, p.z] for p in hand])
    points = points - points[0]
    scale = np.max(np.abs(points))
    points = points / scale
    return points.flatten()

## 4. Download the dataset (Kaggle: ASL Alphabet)
Create a token at kaggle.com -> Settings -> API. Paste it when asked (the box is hidden). Never share or commit your token.

In [ ]:
!pip install -q --upgrade kaggle
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste your Kaggle token: ')

In [ ]:
!kaggle datasets download -d grassknoted/asl-alphabet -q
!unzip -q asl-alphabet.zip -d asl_data

In [ ]:
DATA_DIR = 'asl_data/asl_alphabet_train'
if os.path.isdir(DATA_DIR + '/asl_alphabet_train'):
    DATA_DIR = DATA_DIR + '/asl_alphabet_train'
print(DATA_DIR)

for c in sorted(os.listdir(DATA_DIR)):
    print(c, len(os.listdir(os.path.join(DATA_DIR, c))))

## 5. Extract landmarks
500 random images per sign. The `nothing` class (no hand) is skipped; images where no hand is detected are skipped.

In [ ]:
import random

classes = [c for c in sorted(os.listdir(DATA_DIR)) if c != 'nothing']
print(len(classes), classes)

IMAGES_PER_CLASS = 500
random.seed(42)

In [ ]:
X = []
y = []

for label, c in enumerate(classes):
    folder = os.path.join(DATA_DIR, c)
    all_files = os.listdir(folder)
    chosen = random.sample(all_files, IMAGES_PER_CLASS)
    found = 0
    for f in chosen:
        feats = get_landmarks(os.path.join(folder, f))
        if feats is not None:
            X.append(feats)
            y.append(label)
            found += 1
    print(c, 'hands found:', found, '/', IMAGES_PER_CLASS)

In [ ]:
X = np.array(X)
y = np.array(y)
print(X.shape, y.shape)

np.savez('landmarks.npz', X=X, y=y, classes=classes)

## 6. Train / test split and model

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(classes)

model = keras.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32
)

## 7. Evaluation

In [ ]:
from sklearn.metrics import classification_report

pred_probs = model.predict(X_test)
y_pred = np.argmax(pred_probs, axis=1)
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 9))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(num_classes), classes, rotation=90)
plt.yticks(range(num_classes), classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix (test set)')
plt.colorbar()
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='test')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='test')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 8. Save the model and class names

In [ ]:
import json

model.save('sign_model.keras')
json.dump(classes, open('classes.json', 'w'))

from google.colab import files
files.download('sign_model.keras')
files.download('classes.json')
files.download('landmarks.npz')
files.download('confusion_matrix.png')

## 9. Predict from a webcam photo (Colab)
Colab cannot open a webcam directly, so the browser captures one photo. For continuous real-time recognition run `realtime_sign_recognition.py` on a computer.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const capture = document.createElement('button');
          capture.textContent = 'Capture';
          div.appendChild(capture);

          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});

          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();

          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

          await new Promise((resolve) => capture.onclick = resolve);

          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

In [ ]:
def predict_sign(image_path):
    feats = get_landmarks(image_path)
    if feats is None:
        return None, 0.0
    probs = model.predict(feats.reshape(1, -1), verbose=0)[0]
    best = int(np.argmax(probs))
    return classes[best], float(probs[best])

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

photo = take_photo('my_sign.jpg')
letter, conf = predict_sign(photo)
print('Predicted:', letter, '| confidence:', round(conf * 100, 1), '%')
cv2_imshow(cv2.imread(photo))